# 01 – Data Preparation

This notebook loads raw node features, node2vec embeddings, and struc2vec embeddings
for a chosen dataset, combines them into a single DataFrame, and saves it to `data/04_models/`
for use in notebooks 02–04.

## Datasets available
| Name | Description |
|------|-------------|
| `facebook` | Facebook ego-graph (default) |
| `amazon` | Amazon co-purchase network |
| `grid` | Power grid network |
| `lastfm` | LastFM social graph |
| `reddit` | Reddit post network |
| `twitch` | Twitch social network |
| `x3` … `x7` | Synthetic ABCD+o graphs |

## What are the features?
Each node has two groups of features:

**Community-free graph features** (computed without community information):
- `lcc` – local clustering coefficient
- `bc` – betweenness centrality
- `cc` – closeness centrality
- `dc` – degree centrality
- `ndc` – normalised degree centrality
- `ec` – eigenvector centrality
- `eccen` – eccentricity
- `core` – core number (k-core decomposition)

**Community-aware features** (require knowledge of communities – used as "cheating" baseline):
- `participation` – participation coefficient (how spread are node's edges across communities)
- `in_mod_deg` – intra-community degree

**Regression targets** (13 continuous community measures):
`cada`, `edge_contr`, `in_mod_deg`, `participation`, `beta_star`,
`l1`, `l2`, `kl`, `hd`, `l12`, `l22`, `kl2`, `hd2`

**Classification target**: `target` (binary: 1 = overlapping/community node, 0 = otherwise)

In [ ]:
# ── CHANGE THIS to switch dataset ──────────────────────────────────────────
DATASET = "facebook"
# ───────────────────────────────────────────────────────────────────────────

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.impute import SimpleImputer

ROOT = Path("..")

# Paths derived from dataset name
PATH_FEATURES  = ROOT / f"data/01_raw/data/{DATASET}.parquet"
PATH_EDGES     = ROOT / f"data/01_raw/edges/edge_{DATASET}.parquet"
PATH_N2V       = ROOT / f"data/03_embeddings/{DATASET}_node2vec_embedding.parquet"
PATH_S2V       = ROOT / f"struc2vec/emb/{DATASET}.emb"
PATH_OUTPUT    = ROOT / f"data/04_models/{DATASET}_combined.parquet"

print(f"Dataset: {DATASET}")
print(f"Features : {PATH_FEATURES}")
print(f"Edges    : {PATH_EDGES}")
print(f"Node2Vec : {PATH_N2V}")
print(f"Struc2Vec: {PATH_S2V}")

## Step 1 – Load node features

In [ ]:
features = pd.read_parquet(PATH_FEATURES)

print(f"Shape: {features.shape}")
print(f"Columns: {features.columns.tolist()}")
features.head()

## Step 2 – Load and inspect edges

The edge list has two columns: `in` and `out` (source and target node IDs).
We normalise node IDs to be 0-based (Julia uses 1-based indexing).

In [ ]:
edges = pd.read_parquet(PATH_EDGES)

# Normalise to 0-based if needed
if 0 not in set(edges["in"]) and 0 not in set(edges["out"]):
    print("Edge IDs start from 1 → converting to 0-based")
    edges = edges - 1
else:
    print("Edge IDs already 0-based")

print(f"Edges shape: {edges.shape}")
print(f"Node ID range: [{edges.min().min()} … {edges.max().max()}]")
edges.head()

## Step 3 – Load node2vec embeddings

Node2Vec maps each node to a 16-dimensional vector by simulating biased random walks
and training a Word2Vec model on those walks.
Parameters used: `dim=16`, `walk_length=50`, `window=10`, `p=1`, `q=1`.

The embeddings were pre-computed with `fastnode2vec` and stored as parquet.
Row index = node ID.

In [ ]:
n2v = pd.read_parquet(PATH_N2V)

print(f"Shape: {n2v.shape}  (nodes × embedding dimensions)")
print(f"Columns: {n2v.columns.tolist()}")
n2v.head(3)

## Step 4 – Load struc2vec embeddings

Struc2Vec captures **structural identity** – nodes with similar local graph structure
get similar embeddings, even if they are far apart in the graph.

The `.emb` file format (Word2Vec output):
- Line 1: `num_nodes num_dimensions`
- Remaining lines: `node_id  dim_1  dim_2  …  dim_16`

In [ ]:
def load_struc2vec(path: Path) -> pd.DataFrame:
    """Load struc2vec .emb file, sort by node_id, return embedding columns only."""
    s2v_raw = pd.read_csv(path, sep=" ", skiprows=1, header=None)

    n_dims = s2v_raw.shape[1] - 1  # first column is node_id
    col_names = ["node_id"] + [f"emb_struc2vec_{i}" for i in range(1, n_dims + 1)]
    s2v_raw.columns = col_names

    # Sort by node_id so rows align with features DataFrame
    s2v_raw = s2v_raw.sort_values("node_id").reset_index(drop=True)
    s2v_raw = s2v_raw.drop(columns=["node_id"])
    return s2v_raw

s2v = load_struc2vec(PATH_S2V)

print(f"Shape: {s2v.shape}  (nodes × embedding dimensions)")
s2v.head(3)

## Step 5 – Combine features and embeddings

In [ ]:
# Concatenate horizontally: node features | node2vec | struc2vec
combined = pd.concat([features, n2v, s2v], axis="columns")

print(f"Combined shape: {combined.shape}")
print(f"Total columns: {len(combined.columns)}")
combined.head(3)

## Step 6 – Handle missing values

Nodes without a community assignment have `NaN` in the `target` column.
We drop those rows (they can't be used for classification).
Remaining NaNs in features are filled with 0.

In [ ]:
print(f"Rows before NaN drop: {len(combined)}")
print(f"NaNs in 'target': {combined['target'].isna().sum()}")

combined = combined.dropna(subset=["target"]).reset_index(drop=True)

# Fill remaining NaNs with 0 (e.g. eigenvector centrality on isolated nodes)
imputer = SimpleImputer(strategy="constant", fill_value=0)
combined = pd.DataFrame(
    imputer.fit_transform(combined),
    columns=combined.columns
)

print(f"Rows after NaN drop : {len(combined)}")
print(f"Remaining NaNs total: {combined.isna().sum().sum()}")

## Step 7 – Summary and save

In [ ]:
COMMUNITY_FREE = ["lcc", "bc", "cc", "dc", "ndc", "ec", "eccen", "core"]
COMMUNITY_AWARE = ["participation", "in_mod_deg"]
REGRESSION_TARGETS = ["cada", "edge_contr", "in_mod_deg", "participation",
                      "beta_star", "l1", "l2", "kl", "hd", "l12", "l22", "kl2", "hd2"]
N2V_COLS  = [c for c in combined.columns if c.startswith("emb_node2vec")]
S2V_COLS  = [c for c in combined.columns if c.startswith("emb_struc2vec")]

print(f"Dataset            : {DATASET}")
print(f"Nodes              : {len(combined)}")
print(f"Community-free     : {len(COMMUNITY_FREE)} features")
print(f"Community-aware    : {len(COMMUNITY_AWARE)} features (cheating baseline)")
print(f"Node2Vec dims      : {len(N2V_COLS)}")
print(f"Struc2Vec dims     : {len(S2V_COLS)}")
print(f"Regression targets : {len(REGRESSION_TARGETS)}")
print(f"Classification     : 'target' (binary)")
print(f"\nClass balance:\n{combined['target'].value_counts()}")

combined.to_parquet(PATH_OUTPUT, index=False)
print(f"\nSaved to: {PATH_OUTPUT}")